# FASTQ to counts
This notebook shows how to transform transcriptomics FASTQ files to generate count files which are typically later analyzed in Python/R.

We will be using a dataset of human embryonic stem cells 8/14 days old. Sequenced with 10x Chromium.

We will download all the FASTQ files,
Perform QC with FastQC, and observe the results with MultiQC
Determine if trimming is needed (spoiler: it's not),
Align the reads to the genome with STARsolo to assess data quality,
and align the reads to the transcriptome to generate the counts file. 

Even though we are using lightweight tools, running this notebook locally requires X GB of space (X for FASTQs, Y for STAR genome, Z for transcriptome),
and 1234 GB RAM.

If space and memory were not concerns (e.g., running in HPC), then Cell Ranger would have been a better option for genome and transcriptome alignment because this is 10x data.

This tutorial assumes you have these tools installed in your environment.

In [ ]:
%%bash
set -euo pipefail
# Download the E-MTAB-13632 10x Genomics scRNA-seq dataset from ArrayExpress.

download() {
    local data_dir="${1:-data}"
    local accession="E-MTAB-13632"
    local out_dir="${data_dir}/${accession}"
    local sdrf_url="https://www.ebi.ac.uk/biostudies/files/${accession}/${accession}.sdrf.txt"
    local sdrf_file="${out_dir}/${accession}.sdrf.txt"
    local urls_file="${out_dir}/read_fastq_urls.txt"
    local fastq_dir="${out_dir}/fastq"

    mkdir -p "${fastq_dir}"

    # Download the SDRF sample sheet.
    curl -L "${sdrf_url}" -o "${sdrf_file}"

    # Extract only biological read FASTQs.
    # For 10x scRNA-seq counting, we want R1 + R2 and do not need I1/I2 here.
    tr '\t' '\n' < "${sdrf_file}" \
        | grep -Eo 'ftp://[^[:space:]]+_(R1|R2)_001\.fastq\.gz' \
        | sort -u > "${urls_file}"

    printf "FASTQ URLs to download: %s\n" "$(wc -l < "${urls_file}")"

    # Download R1/R2 FASTQs in parallel.
    xargs -a "${urls_file}" -n 1 -P 4 wget -c -P "${fastq_dir}"
}

download_if_missing() {
    local data_dir="${1:-data}"
    local accession="E-MTAB-13632"
    local out_dir="${data_dir}/${accession}"
    local fastq_dir="${out_dir}/fastq"

    # Treat the download as complete only if we already have reads.
    if compgen -G "${fastq_dir}/*_R1_001.fastq.gz" > /dev/null && \
       compgen -G "${fastq_dir}/*_R2_001.fastq.gz" > /dev/null; then
        printf "R1/R2 FASTQs already present in %s\n" "${fastq_dir}"
    else
        download "${data_dir}"
    fi
}

download_if_missing "${1:-data}"

R1/R2 FASTQs already present in data/E-MTAB-13632/fastq


Before alignment, we should inspect raw read quality, adapter content, duplication, and overrepresented sequences. We will use FastQC for per-sample quality control and MultiQC to combine all reports into one place. This helps decide whether trimming is necessary and if the quality is sufficient before generating counts.

In [2]:
%%bash
set -euo pipefail
# Generate a sample sheet for easy downstream processing.

data_dir="data/E-MTAB-13632"
fastq_dir="${data_dir}/fastq"
sample_sheet="${data_dir}/samples.tsv"

find "${fastq_dir}" -maxdepth 1 -type f -name "*_R1_*.fastq.gz" | sort \
| awk 'BEGIN{OFS="\t"; print "sample_id","read1","read2"}
{
    r1=$0
    r2=$0
    gsub("_R1_","_R2_",r2)

    file=$0
    sub(".*/","",file)

    sample=file
    sub("_R1_001.fastq.gz$","",sample)

    print sample, r1, r2
}' > "${sample_sheet}"

column -ts $'\t' "${sample_sheet}" | head -n 10

sample_id        read1                                                    read2
SITTE1_S4_L001   data/E-MTAB-13632/fastq/SITTE1_S4_L001_R1_001.fastq.gz   data/E-MTAB-13632/fastq/SITTE1_S4_L001_R2_001.fastq.gz
SITTF1_S4_L001   data/E-MTAB-13632/fastq/SITTF1_S4_L001_R1_001.fastq.gz   data/E-MTAB-13632/fastq/SITTF1_S4_L001_R2_001.fastq.gz
SITTG1_S4_L001   data/E-MTAB-13632/fastq/SITTG1_S4_L001_R1_001.fastq.gz   data/E-MTAB-13632/fastq/SITTG1_S4_L001_R2_001.fastq.gz
SITTH10_S4_L001  data/E-MTAB-13632/fastq/SITTH10_S4_L001_R1_001.fastq.gz  data/E-MTAB-13632/fastq/SITTH10_S4_L001_R2_001.fastq.gz


# FastQC
We now run FastQC on all raw FASTQ files. The key outputs to inspect are per-base sequence quality, adapter content, sequence duplication, GC distribution, and overrepresented sequences. In RNA-seq, mild duplication is common for highly expressed transcripts, so the most important early decision point is usually whether adapter contamination or poor tail quality warrants trimming.

In [ ]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
sample_sheet="${data_dir}/samples.tsv"
qc_dir="${data_dir}/qc/raw_fastqc"
threads=8

mkdir -p "${qc_dir}"

# run FastQC on all FASTQ files in parallel. [is parallel always recommended? maybe we should also show the basics]
# extract columns 2,3 (R1 and R2 FASTQ file paths)
# convert tabs to newlines to get a list of all FASTQ files
tail -n +2 "${sample_sheet}" \
| cut -f2,3 \
| tr '\t' '\n' \
| xargs -n 1 -P "${threads}" -I {} fastqc \
    --threads 1 \
    --outdir "${qc_dir}" \
    "{}"

# MultiQC
Now we can merge all the reports with MultiQC

In [1]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
raw_fastqc_dir="${data_dir}/qc/raw_fastqc"
multiqc_dir="${data_dir}/qc/raw_multiqc"

mkdir -p "${multiqc_dir}"

multiqc "${raw_fastqc_dir}" \
    --outdir "${multiqc_dir}" \
    --filename "multiqc_raw_reads"


/// ]8;id=171204;https://multiqc.info\MultiQC]8;;\ 🔍 v1.33

       file_search | Search path: /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/qc/raw_fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0/16  ━━━━━━━━━━━━━━━━━━━━ 100% 16/16  
            fastqc | Found 8 reports
     write_results | Data        : data/E-MTAB-13632/qc/raw_multiqc/multiqc_raw_reads_data
     write_results | Report      : data/E-MTAB-13632/qc/raw_multiqc/multiqc_raw_reads.html
           multiqc | MultiQC complete


Now we can open /qc/raw_multiqc/multiqc_raw_reads.html and observe the quality and need for trimming.

[IMAGE]

Here the only concern would be the lower sequencing depth for the last 2 libraries (SITTH10).
Duplicates in scRNA at these levels are typically tolerated, since more abundant transcripts get preferentially sequenced.

# STARsolo
We will generate a gene-by-cell count matrix from 10x-style single-cell FASTQ files using STARsolo. STARsolo performs alignment, cell barcode correction, UMI deduplication, and gene-level quantification from raw FASTQ files. For this dataset, Read 1 contains the cell barcode and UMI, while Read 2 contains the transcript sequence that will be aligned to the human reference genome.

This uses the 10x Genomics GRCh38 reference package because it already has a Cell Ranger-style FASTA and filtered GTF. 10x’s own installation docs describe refdata-gex-GRCh38-2020-A as a human GRCh38 reference package used by Cell Ranger.

In [3]:
%%bash
set -euo pipefail

project_dir="$(pwd)"
ref_dir="${project_dir}/data/E-MTAB-13632/reference"
source_dir="${ref_dir}/source"

mkdir -p "${source_dir}"

fasta_gz="${source_dir}/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz"
gtf_gz="${source_dir}/gencode.v32.primary_assembly.annotation.gtf.gz"
whitelist_gz="${source_dir}/3M-february-2018.txt.gz"

genome_fasta="${ref_dir}/genome.fa"
genes_gtf="${ref_dir}/genes.gtf"
genes_filtered_gtf="${ref_dir}/genes.filtered.gtf"
whitelist="${ref_dir}/3M-february-2018.txt"

wget -c \
    -O "${fasta_gz}" \
    "https://ftp.ensembl.org/pub/release-98/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz"

wget -c \
    -O "${gtf_gz}" \
    "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/gencode.v32.primary_assembly.annotation.gtf.gz"

wget -c \
    -O "${whitelist_gz}" \
    "https://raw.githubusercontent.com/10XGenomics/cellranger/a83c753ce641db6409a59ad817328354fbe7187e/lib/python/cellranger/barcodes/3M-february-2018.txt.gz"

# Convert Ensembl FASTA headers to GENCODE/Cell-Ranger-style chromosome names.
# Example: >1 ... becomes >chr1 1, and >MT ... becomes >chrM MT.
gzip -dc "${fasta_gz}" \
| sed -E 's/^>(\S+).*/>\1 \1/' \
| sed -E 's/^>([0-9]+|[XY]) />chr\1 /' \
| sed -E 's/^>MT />chrM /' \
> "${genome_fasta}"

# Remove version suffixes from gene/transcript/exon IDs, matching the 10x reference build notes.
ID='(ENS(MUS)?[GTE][0-9]+)\.([0-9]+)'

gzip -dc "${gtf_gz}" \
| sed -E 's/gene_id "'"${ID}"'";/gene_id "\1"; gene_version "\3";/' \
| sed -E 's/transcript_id "'"${ID}"'";/transcript_id "\1"; transcript_version "\3";/' \
| sed -E 's/exon_id "'"${ID}"'";/exon_id "\1"; exon_version "\3";/' \
> "${genes_gtf}"

gzip -dc "${whitelist_gz}" > "${whitelist}"

ls -lh \
    "${genome_fasta}" \
    "${genes_gtf}" \
    "${whitelist}"

--2026-05-07 21:49:36--  https://ftp.ensembl.org/pub/release-98/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 881211416 (840M) [application/x-gzip]
Saving to: ‘/home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/source/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz’

     0K .......... .......... .......... .......... ..........  0%  175K 81m53s
    50K .......... .......... .......... .......... ..........  0% 1.12M 47m13s
   100K .......... .......... .......... .......... ..........  0%  477K 41m30s
   150K .......... .......... .......... .......... ..........  0% 1.27M 33m53s
   200K .......... .......... .......... .......... ..........  0%  511K 32m43s
   250K .......... .......... .......... ...

-rw-r--r-- 1 dekel dekel 111M May  7 21:51 /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/3M-february-2018.txt
-rw-r--r-- 1 dekel dekel 1.4G May  7 21:51 /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/genes.gtf
-rw-r--r-- 1 dekel dekel 3.0G May  7 21:50 /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/genome.fa


In [5]:
%%bash
set -euo pipefail

project_dir="$(pwd)"

ref_dir="${project_dir}/data/E-MTAB-13632/reference"
genes_gtf="${ref_dir}/genes.gtf"
genes_filtered_gtf="${ref_dir}/genes.filtered.gtf"
gene_allowlist="${ref_dir}/gene_allowlist.txt"

BIOTYPE_PATTERN="(protein_coding|lncRNA|IG_C_gene|IG_D_gene|IG_J_gene|IG_LV_gene|IG_V_gene|IG_V_pseudogene|IG_J_pseudogene|IG_C_pseudogene|TR_C_gene|TR_D_gene|TR_J_gene|TR_V_gene|TR_V_pseudogene|TR_J_pseudogene)"
GENE_PATTERN="gene_type \"${BIOTYPE_PATTERN}\""
TX_PATTERN="transcript_type \"${BIOTYPE_PATTERN}\""
READTHROUGH_PATTERN="tag \"readthrough_transcript\""
PAR_PATTERN="tag \"PAR\""

awk '$3 == "transcript"' "${genes_gtf}" \
| grep -E "${GENE_PATTERN}" \
| grep -E "${TX_PATTERN}" \
| grep -Ev "${READTHROUGH_PATTERN}" \
| grep -Ev "${PAR_PATTERN}" \
| sed -E 's/.*(gene_id "[^"]+").*/\1/' \
| sort \
| uniq \
> "${gene_allowlist}"

awk '/^#/' "${genes_gtf}" > "${genes_filtered_gtf}"

grep -Ff "${gene_allowlist}" "${genes_gtf}" >> "${genes_filtered_gtf}"

printf "Genes in allowlist:\n"
wc -l "${gene_allowlist}"

printf "Filtered GTF features:\n"
grep -vc '^#' "${genes_filtered_gtf}"

ls -lh "${genes_filtered_gtf}"

Genes in allowlist:
36601 /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/gene_allowlist.txt
Filtered GTF features:
2765969
-rw-r--r-- 1 dekel dekel 1.4G May  7 21:53 /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/genes.filtered.gtf


In [ ]:
%%bash
set -euo pipefail
#------------------------------------------------------------------------------------------------------------#
project_dir="$(pwd)"

ref_dir="${project_dir}/data/E-MTAB-13632/reference"
star_index="${ref_dir}/star_index_sparseD10"

genome_fasta="${ref_dir}/genome.fa"
genes_filtered_gtf="${ref_dir}/genes.filtered.gtf"

threads=2
sjdb_overhang=89

mkdir -p "${star_index}"

# use free -h to check available RAM and adjust --limitGenomeGenerateRAM accordingly
STAR \
    --runThreadN "${threads}" \
    --runMode genomeGenerate \
    --genomeDir "${star_index}" \
    --genomeFastaFiles "${genome_fasta}" \
    --sjdbGTFfile "${genes_filtered_gtf}" \
    --sjdbOverhang "${sjdb_overhang}" \
    --genomeSAsparseD 10 \
    --limitGenomeGenerateRAM 12000000000

	/home/dekel/miniforge3/envs/scrna-prep/bin/STAR-avx2 --runThreadN 2 --runMode genomeGenerate --genomeDir /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/star_index_sparseD3 --genomeFastaFiles /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/genome.fa --sjdbGTFfile /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/reference/genes.filtered.gtf --sjdbOverhang 89 --genomeSAsparseD 10 --limitGenomeGenerateRAM 12000000000
	STAR version: 2.7.11b   compiled: 2025-07-24T03:06:21+0000 :/opt/conda/conda-bld/star_1753326220084/work/source
May 13 21:58:42 ..... started STAR run
May 13 21:58:42 ... starting to generate Genome files
May 13 21:59:13 ..... processing annotations GTF
May 13 21:59:20 ... starting to sort Suffix Array. This may take a long time...
May 13 21:59:22 ... sorting Suffix Array chunks an

In [ ]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
reference_dir="${data_dir}/reference"

sample_sheet="${data_dir}/samples.tsv"
star_index="${reference_dir}/star_index_sparseD10"
whitelist="${reference_dir}/3M-february-2018.txt"

starsolo_dir="${data_dir}/starsolo"
mkdir -p "${starsolo_dir}"

threads=2

tail -n +2 "${sample_sheet}" \
| while IFS=$'\t' read -r sample_id read1 read2 expected_cells; do

    out_dir="${starsolo_dir}/${sample_id}"
    mkdir -p "${out_dir}"

    echo "Running STARsolo for ${sample_id}"

    STAR \
        --runThreadN "${threads}" \
        --genomeDir "${star_index}" \
        --readFilesIn "${read2}" "${read1}" \
        --readFilesCommand zcat \
        --outFileNamePrefix "${out_dir}/" \
        --outSAMtype None \
        --soloType CB_UMI_Simple \
        --soloUMIlen 12 \
        --soloCBwhitelist "${whitelist}" \
        --soloCBmatchWLtype 1MM_multi_Nbase_pseudocounts \
        --soloUMIfiltering MultiGeneUMI_CR \
        --soloUMIdedup 1MM_CR \
        --clipAdapterType CellRanger4 \
        --outFilterScoreMin 30 \
        --soloCellFilter None

done

Running STARsolo for SITTE1_S4_L001
	/home/dekel/miniforge3/envs/scrna-prep/bin/STAR-avx2 --runThreadN 2 --genomeDir data/E-MTAB-13632/reference/star_index_sparseD10 --readFilesIn data/E-MTAB-13632/fastq/SITTE1_S4_L001_R2_001.fastq.gz data/E-MTAB-13632/fastq/SITTE1_S4_L001_R1_001.fastq.gz --readFilesCommand zcat --outFileNamePrefix data/E-MTAB-13632/starsolo/SITTE1_S4_L001/ --outSAMtype None --soloType CB_UMI_Simple --soloUMIlen 12 --soloCBwhitelist data/E-MTAB-13632/reference/3M-february-2018.txt --soloCBmatchWLtype 1MM_multi_Nbase_pseudocounts --soloUMIfiltering MultiGeneUMI_CR --soloUMIdedup 1MM_CR --clipAdapterType CellRanger4 --outFilterScoreMin 30 --soloCellFilter None
	STAR version: 2.7.11b   compiled: 2025-07-24T03:06:21+0000 :/opt/conda/conda-bld/star_1753326220084/work/source
May 16 21:49:38 ..... started STAR run
May 16 21:49:39 ..... loading genome
May 16 21:49:47 ..... started mapping


In [2]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
sample_sheet="${data_dir}/samples.tsv"

reference_dir="${data_dir}/reference"
star_index="${reference_dir}/STAR_GRCh38_2020A_sparseD10"
whitelist="${reference_dir}/3M-february-2018.txt"

starsolo_dir="${data_dir}/starsolo"
mkdir -p "${starsolo_dir}"

threads=6

tail -n +2 "${sample_sheet}" \
| while IFS=$'\t' read -r sample_id read1 read2 expected_cells; do

    out_dir="${starsolo_dir}/${sample_id}"
    mkdir -p "${out_dir}"

    STAR \
        --runThreadN "${threads}" \
        --genomeDir "${star_index}" \
        --readFilesIn "${read2}" "${read1}" \
        --readFilesCommand zcat \
        --outFileNamePrefix "${out_dir}/" \
        --outSAMtype None \
        --soloType CB_UMI_Simple \
        --soloCBstart 1 \
        --soloCBlen 16 \
        --soloUMIstart 17 \
        --soloUMIlen 12 \
        --soloCBwhitelist "${whitelist}" \
        --soloCBmatchWLtype 1MM_multi_Nbase_pseudocounts \
        --soloUMIfiltering MultiGeneUMI_CR \
        --soloUMIdedup 1MM_CR \
        --soloCellFilter EmptyDrops_CR "${expected_cells}" 0.99 10 45000 90000 500 0.01 20000 0.001 10000 \
        --soloStrand Forward \
        --soloFeatures Gene \
        --clipAdapterType CellRanger4 \
        --outFilterScoreMin 30 \
        --genomeLoad NoSharedMemory

done


EXITING because of fatal PARAMETERS error: --soloCellFilterType EmptyDrops_CR requires exactly 10 numerical parameters
SOLUTION: re-run with --soloCellFilterType EmptyDrops_CR <nExpectedCells> <maxPercentile> <maxMinRatio> <indMin> <indMax> <umiMin> <umiMinFracMedian> <candMaxN> <FDR> <simN>

May 16 21:48:03 ...... FATAL ERROR, exiting


CalledProcessError: Command 'b'set -euo pipefail\n\ndata_dir="data/E-MTAB-13632"\nsample_sheet="${data_dir}/samples.tsv"\n\nreference_dir="${data_dir}/reference"\nstar_index="${reference_dir}/STAR_GRCh38_2020A_sparseD10"\nwhitelist="${reference_dir}/3M-february-2018.txt"\n\nstarsolo_dir="${data_dir}/starsolo"\nmkdir -p "${starsolo_dir}"\n\nthreads=6\n\ntail -n +2 "${sample_sheet}" \\\n| while IFS=$\'\\t\' read -r sample_id read1 read2 expected_cells; do\n\n    out_dir="${starsolo_dir}/${sample_id}"\n    mkdir -p "${out_dir}"\n\n    STAR \\\n        --runThreadN "${threads}" \\\n        --genomeDir "${star_index}" \\\n        --readFilesIn "${read2}" "${read1}" \\\n        --readFilesCommand zcat \\\n        --outFileNamePrefix "${out_dir}/" \\\n        --outSAMtype None \\\n        --soloType CB_UMI_Simple \\\n        --soloCBstart 1 \\\n        --soloCBlen 16 \\\n        --soloUMIstart 17 \\\n        --soloUMIlen 12 \\\n        --soloCBwhitelist "${whitelist}" \\\n        --soloCBmatc

In [ ]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"

find "${data_dir}/starsolo" \
    -path "*/Solo.out/Gene/filtered/matrix.mtx" \
    -print | sort



starsolo_dir="data/E-MTAB-13632/starsolo"

find "${starsolo_dir}" \
    -path "*/Solo.out/Gene/*" \
    \( -name "matrix.mtx" -o -name "barcodes.tsv" -o -name "features.tsv" \) \
    -exec gzip -f {} \;

In [ ]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
sample_sheet="${data_dir}/samples_starsolo.tsv"
starsolo_dir="${data_dir}/starsolo"
summary_tsv="${data_dir}/starsolo_summary.tsv"

printf "sample_id\tmetric\tvalue\n" > "${summary_tsv}"

tail -n +2 "${sample_sheet}" \
| while IFS=$'\t' read -r sample_id read1 read2 expected_cells; do
    summary_csv="${starsolo_dir}/${sample_id}/Solo.out/Gene/Summary.csv"

    awk -F',' -v sample_id="${sample_id}" '
        NF >= 2 {
            metric = $1
            value = $2

            gsub(/^[ \t]+|[ \t]+$/, "", metric)
            gsub(/^[ \t]+|[ \t]+$/, "", value)

            print sample_id "\t" metric "\t" value
        }
    ' "${summary_csv}" >> "${summary_tsv}"
done

column -ts $'\t' "${summary_tsv}" | sed -n '1,160p'



data_dir="data/E-MTAB-13632"
starsolo_dir="${data_dir}/starsolo"
multiqc_dir="${data_dir}/qc/starsolo_multiqc"

mkdir -p "${multiqc_dir}"

multiqc "${starsolo_dir}" \
    --outdir "${multiqc_dir}" \
    --filename "multiqc_starsolo"